# ACT Walkthrough

This notebook walks through the one-file ACT demo in the order the model sees the world:

1. download a small real robomimic dataset
2. inspect the raw HDF5 layout
3. flatten robomimic observations into state vectors
4. build action chunks
5. run one ACT forward pass
6. train a tiny run and inspect the outputs


## 1. Imports

Everything important lives in `act.py`. The notebook imports those pieces instead of duplicating the implementation.

In [1]:
from pathlib import Path
import h5py
import torch

from act import ACT, DemoDataset, download_dataset, sorted_demo_keys, split_loaders, state_from_robomimic_obs

print("imports ok")


imports ok


## 2. Download a real dataset

`lift-ph` is a small robomimic low-dimensional dataset. It is real demonstration data, but small enough to use for local learning and smoke tests.

In [2]:
path = download_dataset("lift-ph", out=None, force=False)
print("dataset path:", path)
print("file size MB:", round(path.stat().st_size / 1024 / 1024, 1))


exists: data/lift_ph_low_dim.hdf5
dataset path: data/lift_ph_low_dim.hdf5
file size MB: 20.1


## 3. Inspect the HDF5 layout

robomimic stores demonstrations under `/data/demo_*`. Each demo has `actions` and an `obs` group. In this repo, every low-dimensional observation array in `obs` becomes part of the state vector.

In [3]:
with h5py.File(path, "r") as f:
    demos = sorted_demo_keys(f["data"].keys())
    demo = f["data"][demos[0]]
    print("first demos:", demos[:5])
    print("actions shape:", demo["actions"].shape)
    print("\nobs arrays:")
    for key in sorted(demo["obs"].keys()):
        print(f"  {key:<24} {demo['obs'][key].shape}")


first demos: ['demo_0', 'demo_1', 'demo_2', 'demo_3', 'demo_4']
actions shape: (59, 7)

obs arrays:
  object                   (59, 10)
  robot0_eef_pos           (59, 3)
  robot0_eef_quat          (59, 4)
  robot0_eef_quat_site     (59, 4)
  robot0_gripper_qpos      (59, 2)
  robot0_gripper_qvel      (59, 2)
  robot0_joint_pos         (59, 7)
  robot0_joint_pos_cos     (59, 7)
  robot0_joint_pos_sin     (59, 7)
  robot0_joint_vel         (59, 7)


## 4. Build one state vector

At each timestep, ACT needs one fixed-size observation vector. `state_from_robomimic_obs` concatenates all 2D low-dimensional observation arrays at timestep `t`.

In [4]:
with h5py.File(path, "r") as f:
    obs = f["data/demo_0/obs"]
    state = state_from_robomimic_obs(obs, t=0)

print("state shape at t=0:", state.shape)
print("first 8 values:", state[:8].round(3))


state shape at t=0: (53,)
first 8 values: [-0.042  0.011  0.841  1.    -0.     0.     0.     0.   ]


## 5. Build action-chunk samples

ACT predicts a short future sequence of actions. If `chunk_size = 8`, one sample contains the current state and the next 8 actions.

In [5]:
dataset = DemoDataset(path, chunk_size=8, max_demos=10)
sample = dataset[0]

print("demos used:", len(dataset.demos))
print("training samples:", len(dataset))
print("state_dim:", dataset.state_dim)
print("action_dim:", dataset.action_dim)
print("sample['state'].shape:", tuple(sample["state"].shape))
print("sample['actions'].shape:", tuple(sample["actions"].shape))


demos used: 10
training samples: 461
state_dim: 53
action_dim: 7
sample['state'].shape: (53,)
sample['actions'].shape: (8, 7)


## 6. Batch the data

The model never sees individual Python dicts. It sees two tensors: `(batch, state_dim)` and `(batch, chunk_size, action_dim)`.

In [6]:
full, train_loader, val_loader = split_loaders(path, chunk_size=8, batch_size=64, seed=0, max_demos=10)
batch = next(iter(train_loader))

print("batch state shape:", tuple(batch["state"].shape))
print("batch actions shape:", tuple(batch["actions"].shape))


batch state shape: (64, 53)
batch actions shape: (64, 8, 7)


## 7. One model forward pass

The output shape should match the action chunk target exactly. The latent `mu` and `logvar` are the CVAE-style training latent.

In [7]:
model = ACT(state_dim=full.state_dim, action_dim=full.action_dim, chunk_size=8, dim=64)
pred, mu, logvar = model(batch["state"], batch["actions"])

print("predicted action chunk:", tuple(pred.shape))
print("latent mu:", tuple(mu.shape))
print("latent logvar:", tuple(logvar.shape))


predicted action chunk: (64, 8, 7)
latent mu: (64, 16)
latent logvar: (64, 16)


## 8. Tiny training run

This mirrors the CLI command. One epoch is enough to verify the whole training path, not enough to claim a good policy.

In [8]:
!uv run python act.py train --data data/lift_ph_low_dim.hdf5 --out runs/notebook --epochs 1 --batch-size 64 --chunk-size 8 --dim 64 --max-demos 10
print("wrote: runs/notebook/best.pt")
print("wrote: runs/notebook/metrics.json")


epoch=1 train=6.2127 val=4.5698
wrote: runs/notebook/best.pt
wrote: runs/notebook/metrics.json


## Reading the shapes

- `state_dim = 53`: one flattened low-dimensional observation vector.
- `action_dim = 7`: one continuous robot command.
- `chunk_size = 8`: the model predicts 8 future commands per observation.
- `(64, 8, 7)`: a batch of 64 action chunks, each containing 8 actions of width 7.

That is the core ACT idea in this notebook: predict a chunk of future actions, not one action.